# Exercise 11: Phase-space portraits near resonances

**Block 4: Resonances**

**Learning objectives**
* Draw Poincaré sections (turn-by-turn phase-space plots) for a ring with a single nonlinear kick.
* Recognise the typical pictures: a **triangle** near $Q = 1/3$ (sextupole), **four islands** near
  $Q = 1/4$ (octupole), unbounded growth near $Q = 1/2$ (quadrupole error), and **exchange** or
  **growth** of the two planes near $Q_x = Q_y$ or $Q_x + Q_y = 1$ (skew quadrupole).
* Relate the number of islands to the order of the resonance, and find stable and unstable fixed points.

**Motivation.** In a perfect linear ring every particle moves on a circle in normalised phase space.
A single nonlinear magnet changes this picture completely *if the tune is close to a resonance*,
i.e. if after a few turns the particle comes back to almost the same phase and the small kicks
add up instead of averaging out.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

## The machine: a "Hénon-like" ring

`L.HenonRing(Qx, Qy, betx, alfx, bety, alfy, knl, ksl)` is the simplest ring with a nonlinearity:
one turn = a linear rotation with tunes ($Q_x$, $Q_y$) (all the linear optics of the ring rolled into one
matrix) followed by **one thin multipole kick**. We use $\beta_x$ = $\beta_y$ = 4 m and $\alpha$ = 0 at the kick.

We always plot **normalised coordinates** ($\hat{x}$, $\hat{p}$), where the linear motion is a circle.
The action is $J = (\hat{x}^2 + \hat{p}^2)/2 = \varepsilon_i/2$.

For the one-dimensional pictures we start particles with $y$ = $p_y$ = 0. For *normal* multipoles
the vertical kick is zero when $y$ = 0 (check the formula of Exercise 1), so they stay at $y$ = 0
forever: the motion is truly one-dimensional.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, lattices as L, tracking as T, diagnostics as D, plotting as P
from beamlib.plotting import COLORS, MM
from beamlib.widgets import interactive_plot
P.setup()
BETA = 4.0

## 0. Warm-up: the classic Hénon map

In 1969 the astronomer Michel Hénon studied this two-line map:

$p \to p + x^2$, then rotate ($x$, $p$) by the angle $2\pi Q$.

It is exactly our ring with a sextupole, in units where $\beta = 1$. `L.henon_map_2d(x, p, Q)` does one step.

**Task.** Write `iterate_henon`, which applies the map `n` times to arrays of starting points (all particles
at once, no loop over particles) and returns all the turns. Particles that escape (|$x$| > 2)
should become `np.nan`.

In [ ]:
def iterate_henon(x0: np.ndarray, p0: np.ndarray, Q: float, n: int):
    """Arrays x, p of shape (n + 1, len(x0)) with all iterates of the Hénon map."""
    xs, ps = [np.asarray(x0, float)], [np.asarray(p0, float)]
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")
    return np.array(xs), np.array(ps)

In [ ]:
x0 = np.linspace(0.02, 0.9, 30)
X, Pp = iterate_henon(x0, np.zeros_like(x0), Q=0.2114, n=1000)
assert X.shape == (1001, 30)
assert np.isnan(X[-1, -1]) and np.isfinite(X[-1, 0]), "small amplitudes survive, large ones escape"

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X, Pp, c=np.broadcast_to(np.arange(30), X.shape), s=0.3, cmap="viridis")
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("p"); ax.set_title("Hénon map, Q = 0.2114")
plt.show()

Near the centre: circles. Further out: a chain of **5 islands** (the tune there has shifted to 1/5),
then chaos, then escape. All of this from one $x^2$ kick per turn! Now we look at each magnet type separately.

## 1. A tool for all portraits

**Task.** Write `poincare_1d(ring, amplitudes, n_turns, angle=0)`: start one particle per amplitude
$a$ at $\hat{x} = a \cos(\text{angle})$, $\hat{p} = −a \sin(\text{angle})$, $\hat{y}$ = $\hat{p}_y$ = 0 (normalised coordinates; use
`ring.denormalise` to get physical coordinates), track, and return the normalised ($\hat{x}$, $\hat{p}$) arrays of
shape (n_turns + 1, N) (use `ring.normalise` on `res.coords`).

In [ ]:
def poincare_1d(ring, amplitudes, n_turns: int = 1000, angle: float = 0.0):
    """Normalised turn-by-turn coordinates (x_hat, p_hat), each (n_turns + 1, N)."""
    a = np.asarray(amplitudes, dtype=float)
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
lin = L.HenonRing(Qx=0.31, Qy=0.27, betx=BETA, bety=BETA)
xh, ph = poincare_1d(lin, [1e-3, 2e-3], 50)
assert xh.shape == (51, 2)
assert np.allclose(np.hypot(xh, ph), [1e-3, 2e-3]), "without a kick the motion is a circle"
print("✅ poincare_1d works")


lin = L.HenonRing(Qx=0.31, Qy=0.27, betx=BETA, bety=BETA)
xh, ph = poincare_1d(lin, [1e-3, 2e-3], 50)
assert xh.shape == (51, 2)
assert np.allclose(np.hypot(xh, ph), [1e-3, 2e-3]), "without a kick the motion is a circle"
print("GOOD! poincare_1d works")


def find_fixed_points(ring, period, rmax, n_r=10, n_angle=16):
    """Stable and unstable fixed points of `period` turns, in normalised coordinates.

    Newton's method is started from a grid of points; every fixed point found is
    followed for `period` turns to add the other points of the same orbit."""
    stable, unstable = [], []
    for r in np.linspace(0.05, 1.0, n_r) * rmax:
        for th in np.linspace(0, 2 * np.pi, n_angle, endpoint=False):
            z0 = ring.denormalise(np.array([r * np.cos(th), r * np.sin(th), 0, 0]))
            z, tr = O.find_fixed_point(ring.one_turn_map, z0, period=period)
            if z is None:
                continue
            lst = stable if abs(tr) < 2 else unstable
            orbit = z[:, None]
            for _ in range(period):          # the whole orbit: z, M(z), M(M(z)), ...
                zn = ring.normalise(orbit)[:2, 0]
                if 1e-6 < np.hypot(*zn) < rmax and all(np.hypot(*(zn - q)) > 1e-7 for q in lst):
                    lst.append(zn)
                orbit = ring.one_turn_map(orbit)
    return np.array(stable), np.array(unstable)


def mark_fixed_points(ax, stable, unstable):
    if len(stable):
        ax.plot(stable[:, 0] * MM, stable[:, 1] * MM, "o", mfc="none", mec="r", ms=9, mew=2, label="stable")
    if len(unstable):
        ax.plot(unstable[:, 0] * MM, unstable[:, 1] * MM, "x", color="r", ms=9, mew=2, label="unstable")

`find_fixed_points(ring, period, rmax)` (given) searches for points that come back to themselves
after `period` turns, with Newton's method. It classifies them by the trace of the period-turn
Jacobian: |trace| < 2 is **stable** (a particle nearby circles around it: the centre of an island),
|trace| > 2 is **unstable** (a particle nearby is pushed away along a separatrix).

## 2. Sextupole near $Q = 1/3$

**Predict first.** A sextupole with $Q_x = 0.32$. A particle at the resonance comes back to the same phase
every 3 turns. How many fixed points of 3 turns do you expect? What shape will the stable region have?

*Your prediction:*

In [ ]:
ring_s = L.HenonRing(Qx=0.32, Qy=0.27, betx=BETA, bety=BETA, knl=[0, 0, 10.0])
xh, ph = poincare_1d(ring_s, np.linspace(0.5e-3, 11e-3, 16), 1000)
stable_s, unstable_s = find_fixed_points(ring_s, 3, rmax=0.04)
# extra particles started near the stable fixed points, to show the islands around them
isl_x, isl_p = [], []
for zs in stable_s:
    xi, pi = poincare_1d(ring_s, np.hypot(*zs) + np.array([-1.5e-3, -0.8e-3, 0.0]), 1000,
                         angle=np.arctan2(-zs[1], zs[0]))
    isl_x.append(xi); isl_p.append(pi)

fig, axs = plt.subplots(1, 2, figsize=(13, 6))
for ax, lim in zip(axs, [14, 34]):
    ax.scatter(xh * MM, ph * MM, s=0.3, c=np.broadcast_to(np.arange(xh.shape[1]), xh.shape), cmap="viridis")
    for xi, pi in zip(isl_x, isl_p):
        ax.scatter(xi * MM, pi * MM, s=0.3, color=COLORS["accent"])
    mark_fixed_points(ax, stable_s, unstable_s)
    P.phase_space_labels(ax, "x", True); ax.set_aspect("equal")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
axs[0].set_title(r"Sextupole, $Q_x$ = 0.32: zoom on the stable triangle"); axs[1].legend(loc="lower left")
axs[1].set_title("zoomed out: islands around the stable fixed points (orange)")
plt.show()

**Task.** Count the stable and unstable fixed points in the picture and write the numbers below.

In [ ]:
my_stable_sext, my_unstable_sext = None, None
assert (my_stable_sext, my_unstable_sext) == (len(stable_s), len(unstable_s)), "count again (red o and x)"
print(f"GOOD! {len(unstable_s)} unstable fixed points on the triangle, {len(stable_s)} stable ones further out: "
      "third-order resonance -> 3 of each")

The three **unstable** fixed points are the corners of the triangle (the *separatrix*). Inside it the
motion is stable; outside, particles are driven away: this is how a sextupole limits the dynamic
aperture near $Q = 1/3$.

The three **stable** fixed points sit further out, surrounded by small islands. They exist because the
tune changes with amplitude (here a second-order effect of the sextupole): at that amplitude the tune is
exactly 1/3 and the particle stays locked to the resonance. Between the triangle and the islands the
motion is chaotic.

**Explore.** Move $Q_x$ closer to 1/3 and away from it, and change the strength. How does the size of
the triangle depend on $|Q_x − 1/3|$ and on $k_2L$?

In [ ]:
def show_sextupole(Qx, k2l):
    ring = L.HenonRing(Qx=Qx, Qy=0.27, betx=BETA, bety=BETA, knl=[0, 0, k2l])
    xh, ph = poincare_1d(ring, np.linspace(0.5e-3, 20e-3, 25), 600)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(xh * MM, ph * MM, s=0.3, c=np.broadcast_to(np.arange(25), xh.shape), cmap="viridis")
    ax.set_xlim(-22, 22); ax.set_ylim(-22, 22); ax.set_aspect("equal")
    P.phase_space_labels(ax, "x", True); ax.set_title(rf"sextupole $k_2L$ = {k2l} 1/m², $Q_x$ = {Qx:.3f}")
    plt.show()

interactive_plot(show_sextupole, Qx=(0.28, 0.40, 0.002, 0.32), k2l=(1.0, 30.0, 1.0, 10.0))

## 3. Octupole near $Q = 1/4$

**Predict first.** An octupole with $Q_x = 0.24$. How many islands?

*Your prediction:*

In [ ]:
ring_o = L.HenonRing(Qx=0.24, Qy=0.27, betx=BETA, bety=BETA, knl=[0, 0, 0, 1000.0])
stable_o, unstable_o = find_fixed_points(ring_o, 4, rmax=0.025)
xh, ph = poincare_1d(ring_o, np.linspace(0.5e-3, 15e-3, 22), 1000)
th = np.arctan2(-stable_o[0][1], stable_o[0][0])
xh2, ph2 = poincare_1d(ring_o, np.hypot(*stable_o[0]) + np.linspace(-3e-3, 3e-3, 7), 1000, angle=th)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(xh * MM, ph * MM, s=0.3, c=np.broadcast_to(np.arange(22), xh.shape), cmap="viridis")
ax.scatter(xh2 * MM, ph2 * MM, s=0.3, color=COLORS["accent"])
mark_fixed_points(ax, stable_o, unstable_o)
P.phase_space_labels(ax, "x", True); ax.set_aspect("equal"); ax.legend(loc="lower left")
ax.set_xlim(-20, 20); ax.set_ylim(-20, 20)
ax.set_title(r"Octupole, $Q_x$ = 0.24: four islands")
plt.show()

In [ ]:
my_islands_oct = None
assert my_islands_oct == len(stable_o) == len(unstable_o)
print("GOOD! fourth-order resonance -> 4 islands (4 stable + 4 unstable fixed points)")

Here the islands sit at the amplitude where the octupole's **amplitude-dependent tune** reaches exactly
1/4 (Exercise 14). Particles inside an island go around the island centre while jumping from island
to island every turn: their *average* tune is exactly 1/4. Unlike the sextupole triangle, the motion
stays bounded on both sides of the island chain.

In [ ]:
def show_octupole(Qx, k3l):
    ring = L.HenonRing(Qx=Qx, Qy=0.27, betx=BETA, bety=BETA, knl=[0, 0, 0, k3l])
    xh, ph = poincare_1d(ring, np.linspace(0.5e-3, 20e-3, 25), 600)
    xh2, ph2 = poincare_1d(ring, np.linspace(0.5e-3, 20e-3, 25), 600, angle=np.pi / 4)
    fig, ax = plt.subplots(figsize=(6, 6))
    for a, b in [(xh, ph), (xh2, ph2)]:
        ax.scatter(a * MM, b * MM, s=0.3, c=np.broadcast_to(np.arange(25), a.shape), cmap="viridis")
    ax.set_xlim(-25, 25); ax.set_ylim(-25, 25); ax.set_aspect("equal")
    P.phase_space_labels(ax, "x", True); ax.set_title(rf"octupole $k_3L$ = {k3l:.0f} 1/m³, $Q_x$ = {Qx:.3f}")
    plt.show()

interactive_plot(show_octupole, Qx=(0.20, 0.30, 0.002, 0.24), k3l=(-3000.0, 3000.0, 100.0, 1000.0))

Try a negative $k_3L$ with $Q_x = 0.24$: the tune now *decreases* with amplitude, moves away from 1/4,
and the islands disappear. With $Q_x = 0.26$ it is the other way round.

## 4. Quadrupole error near $Q = 1/2$

A thin quadrupole error is *linear*: it does not make the tune depend on amplitude. Near the
half-integer resonance it either just shifts the tune, or, inside the so-called **stopband**, makes the
one-turn matrix unstable.

**Predict first.** $Q_x = 0.49$ with a focusing error $k_1L = +0.1 \text{m}^{-1}$. Will there be islands?

*Your prediction:*

**Task.** Write `is_unstable(Qx, k1l)`: build the ring, take its one-turn matrix (the Jacobian at the
origin, `O.numerical_jacobian(ring.one_turn_map, np.zeros(4))`) and return True if $|\text{Tr}(M_x)|/2 > 1$.

In [ ]:
def is_unstable(Qx: float, k1l: float, beta: float = BETA) -> bool:
    """True if a thin quad error k1l makes the horizontal motion unstable."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
qs = np.linspace(0.40, 0.60, 400)       # (avoids Q = 0.5 exactly, where the edge is marginal)
unst_p = np.array([is_unstable(q, +0.1) for q in qs])
unst_m = np.array([is_unstable(q, -0.1) for q in qs])
# theory: half the trace of (rotation) x (thin quad) is cos(mu) - (beta k1l / 2) sin(mu)
mu = 2 * np.pi * qs
theory = np.abs(np.cos(mu) - BETA * 0.1 / 2 * np.sin(mu)) > 1
assert np.array_equal(unst_p, theory)
print(f"stopband for k1L = +0.1: Qx from {qs[unst_p].min():.3f} to {qs[unst_p].max():.3f}")
print(f"stopband for k1L = -0.1: Qx from {qs[unst_m].min():.3f} to {qs[unst_m].max():.3f}")
print("GOOD! Stopband found")

In [ ]:
ring_q = L.HenonRing(Qx=0.49, Qy=0.27, betx=BETA, bety=BETA, knl=[0, 0.1])
ring_q_ok = L.HenonRing(Qx=0.49, Qy=0.27, betx=BETA, bety=BETA, knl=[0, -0.1])
xh, ph = poincare_1d(ring_q, [1e-4, 2e-4, 3e-4], 60)
xk, pk = poincare_1d(ring_q_ok, [1e-4, 2e-4, 3e-4], 60)

fig, axs = plt.subplots(1, 3, figsize=(16, 4.8))
axs[0].fill_between(qs, 0, unst_p, color=COLORS["y"], alpha=0.4, step="mid", label=r"$k_1L$ = +0.1")
axs[0].fill_between(qs, 0, -unst_m.astype(float), color=COLORS["x"], alpha=0.4, step="mid", label=r"$k_1L$ = -0.1 (shown below 0)")
axs[0].set_xlabel(r"$Q_x$"); axs[0].set_yticks([]); axs[0].legend(fontsize=8); axs[0].set_title("stopbands (unstable regions)")
for j in range(3):
    axs[1].plot(xh[::2, j] * MM, ph[::2, j] * MM, "o", ms=3, color=P.amplitude_colors(3)[j])
    axs[1].plot(xh[1::2, j] * MM, ph[1::2, j] * MM, "s", ms=3, color=P.amplitude_colors(3)[j], alpha=0.5)
axs[1].set_xlim(-30, 30); axs[1].set_ylim(-30, 30); axs[1].set_aspect("equal")
P.phase_space_labels(axs[1], "x", True); axs[1].set_title(r"$Q_x$ = 0.49, $k_1L$ = +0.1: runs away")
J = 0.5 * (xh**2 + ph**2); Jk = 0.5 * (xk**2 + pk**2)
axs[2].semilogy(J[:, 0] / J[0, 0], color=COLORS["y"], label=r"$k_1L$ = +0.1 (in the stopband)")
axs[2].semilogy(Jk[:, 0] / Jk[0, 0], color=COLORS["x"], label=r"$k_1L$ = -0.1 (outside)")
axs[2].set_xlabel("turn"); axs[2].set_ylabel("J / J(0)"); axs[2].legend()
axs[2].set_title("action vs turn")
plt.show()

Inside the stopband the particles run away along a straight line through the origin, jumping to the
opposite side every turn (circles: even turns, squares: odd turns), and the action grows
**exponentially** for every amplitude, however small. There are no islands and no stable fixed point:
since the tune does not depend on amplitude, nothing can stop the growth.

Outside the stopband the motion is stable. $J$ (computed with the $\beta$ of the ring *without* the error)
oscillates because the error changes $\beta$ ("beta-beating"), but it stays bounded.

In [ ]:
def show_halfinteger(Qx, k1l):
    ring = L.HenonRing(Qx=Qx, Qy=0.27, betx=BETA, bety=BETA, knl=[0, k1l])
    xh, ph = poincare_1d(ring, [2e-3, 4e-3, 6e-3], 200)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
    for j in range(3):
        axs[0].plot(xh[:, j] * MM, ph[:, j] * MM, ".", ms=2, color=P.amplitude_colors(3)[j])
    axs[0].set_xlim(-30, 30); axs[0].set_ylim(-30, 30); axs[0].set_aspect("equal")
    P.phase_space_labels(axs[0], "x", True)
    J = 0.5 * (xh**2 + ph**2)
    axs[1].semilogy(J / J[0], lw=1); axs[1].set_xlabel("turn"); axs[1].set_ylabel(r"$J / J(0)$")
    axs[0].set_title(rf"$Q_x$ = {Qx:.3f}, $k_1L$ = {k1l:.3f}: {'UNSTABLE' if is_unstable(Qx, k1l) else 'stable'}")
    plt.show()

interactive_plot(show_halfinteger, Qx=(0.40, 0.60, 0.002, 0.47), k1l=(-0.2, 0.2, 0.01, 0.1))

## 5. Skew quadrupole near $Q_x = Q_y$ (difference resonance)

A skew quadrupole kicks $p_x$ with $y$ and $p_y$ with $x$ (Exercise 1): it **couples** the two planes.
Now the motion is 4D, so instead of a phase-space portrait we look at the actions $J_x$ and $J_y$ turn by turn.

**Predict first.** A particle starts with all its amplitude in $x$ ($J_y$ = 0), with $Q_x$ = 0.310,
$Q_y$ = 0.308. What happens to $J_x$ and J_y$?

*Your prediction:*

In [ ]:
def actions_vs_turn(ring, Jx0, Jy0, n_turns):
    """Track one particle starting at x_hat = sqrt(2 Jx0), y_hat = sqrt(2 Jy0); return Jx, Jy per turn."""
    bn = np.array([[np.sqrt(2 * Jx0)], [0.0], [np.sqrt(2 * Jy0)], [0.0]])   # shape (4, 1)
    res = T.track(ring.denormalise(bn), ring.one_turn_map, n_turns)
    zn = ring.normalise(res.coords)[:, :, 0]
    return 0.5 * (zn[:, 0]**2 + zn[:, 1]**2), 0.5 * (zn[:, 2]**2 + zn[:, 3]**2)

ring_d = L.HenonRing(Qx=0.310, Qy=0.308, betx=BETA, bety=BETA, ksl=[0, 0.015])
Jx, Jy = actions_vs_turn(ring_d, 1e-6, 0.0, 300)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(Jx * 1e6, color=COLORS["x"], label=r"$J_x$")
ax.plot(Jy * 1e6, color=COLORS["y"], label=r"$J_y$")
ax.plot((Jx + Jy) * 1e6, color=COLORS["sum"], lw=2, label=r"$J_x + J_y$")
ax.set_xlabel("turn"); ax.set_ylabel("action [mm mrad]"); ax.legend()
ax.set_title(r"Skew quad near $Q_x = Q_y$: the planes exchange their amplitude")
plt.show()
print(f"max Jy / Jx(0) = {Jy.max() / Jx[0]:.3f};  Jx + Jy constant to {np.ptp(Jx + Jy) / (Jx + Jy).mean() * 100:.1f} %")
assert Jy.max() > 0.9 * Jx[0] and np.ptp(Jx + Jy) / (Jx + Jy).mean() < 0.08

The amplitude flows from $x$ to $y$ and back, like two coupled pendulums. The *sum* $J_x + J_y$ stays
(almost) constant: the total "oscillation energy" is just shared. Nothing is lost, but the beam
becomes tilted and the measured tunes are pulled together.

In [ ]:
def show_difference(Qy, k1s):
    ring = L.HenonRing(Qx=0.310, Qy=Qy, betx=BETA, bety=BETA, ksl=[0, k1s])
    Jx, Jy = actions_vs_turn(ring, 1e-6, 0.0, 400)
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(Jx * 1e6, color=COLORS["x"], label="Jx"); ax.plot(Jy * 1e6, color=COLORS["y"], label="Jy")
    ax.plot((Jx + Jy) * 1e6, color=COLORS["sum"], label="Jx + Jy")
    ax.set_ylim(0, 1.3); ax.set_xlabel("turn"); ax.set_ylabel("action [mm mrad]"); ax.legend(loc="upper right")
    ax.set_title(f"Qx = 0.310, Qy = {Qy:.3f}, k1sL = {k1s:.3f}")
    plt.show()

interactive_plot(show_difference, Qy=(0.25, 0.37, 0.001, 0.308), k1s=(0.0, 0.05, 0.001, 0.015))

Move $Q_y$ away from $Q_x$: the exchange becomes partial and faster. The exchange is complete only if
$|Q_x − Q_y|$ is smaller than the coupling strength $|C| \approx \sqrt{\beta_x \beta_y} k_{1s}L / 2\pi$.

## 6. Skew quadrupole near $Q_x + Q_y = 1$ (sum resonance)

**Predict first.** Same skew quad, but now $Q_x$ = 0.26, $Q_y$ = 0.735 (sum $\approx$ 1). The particle starts with a
tiny amplitude in $x$ only.

*Your prediction:*

In [ ]:
ring_sum = L.HenonRing(Qx=0.26, Qy=0.735, betx=BETA, bety=BETA, ksl=[0, 0.015])
Jx, Jy = actions_vs_turn(ring_sum, 1e-10, 0.0, 300)
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].semilogy(Jx * 1e6, color=COLORS["x"], label=r"$J_x$")
axs[0].semilogy(Jy * 1e6 + 1e-12, color=COLORS["y"], label=r"$J_y$")
axs[0].set_xlabel("turn"); axs[0].set_ylabel("action [mm mrad]"); axs[0].legend()
axs[0].set_title("Sum resonance: both planes grow together")
axs[1].plot((Jx - Jy) / Jx.max(), color=COLORS["sum"], label=r"$(J_x - J_y)$ / max($J_x$)")
axs[1].plot((Jx + Jy) / Jx.max(), color="0.5", ls="--", label=r"$(J_x + J_y)$ / max($J_x$)")
axs[1].set_xlabel("turn"); axs[1].legend(); axs[1].set_title("the difference stays small, the sum explodes")
plt.show()
assert Jx[-1] > 1e4 * Jx[0] and Jy[-1] > 1e4 * Jx[0]

Now $J_x$ − $J_y$ is (nearly) conserved instead of the sum, and both actions grow exponentially together:
the sum resonance is *unstable*, like the half-integer stopband, while the difference resonance only
exchanges amplitude.

In [ ]:
def show_sum(Qy, k1s):
    ring = L.HenonRing(Qx=0.26, Qy=Qy, betx=BETA, bety=BETA, ksl=[0, k1s])
    Jx, Jy = actions_vs_turn(ring, 1e-10, 0.0, 300)
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.semilogy(Jx * 1e6, color=COLORS["x"], label="Jx"); ax.semilogy(Jy * 1e6 + 1e-12, color=COLORS["y"], label="Jy")
    ax.set_xlabel("turn"); ax.set_ylabel("action [mm mrad]"); ax.legend()
    ax.set_title(rf"$Q_x$ = 0.26, $Q_y$ = {Qy:.3f} ($Q_x + Q_y$ = {0.26 + Qy:.3f}), $k_{{1s}}L$ = {k1s:.3f}")
    plt.show()

interactive_plot(show_sum, Qy=(0.68, 0.80, 0.001, 0.735), k1s=(0.0, 0.05, 0.001, 0.015))

## Summary

| magnet | resonance | picture |
|---|---|---|
| normal sextupole | $3Q_x$ = integer | triangle (3 unstable fixed points), 3 islands further out |
| normal octupole | $4Q_x$ = integer | 4 islands, bounded motion |
| normal quadrupole error | $2Q_x$ = integer | no islands; exponential growth inside the stopband |
| skew quadrupole | $Q_x$ − $Q_y$ = integer | exchange of $J_x$ and $J_y$, sum conserved |
| skew quadrupole | $Q_x$ + $Q_y$ = integer | $J_x$ and $J_y$ grow together, difference conserved |

General rule: a resonance $m Q_x + n Q_y$ = integer of order $|m| + |n|$ gives $|m|$ islands in $x$.
A 2($k$+1)-pole drives resonances up to order $k$+1 directly.

## Reflection questions

1. Why do the sextupole and octupole make *islands* while the quadrupole error near 1/2 makes the
   particles run away?
2. Which resonance does a *skew sextupole* drive in the vertical plane? (Try it: `ksl=[0, 0, 10]`, start with $x$ = 0.)
3. If you had to choose the working point of a machine with sextupoles, which tunes would you avoid first?